# Comprueba tu conciliación

Pedidos contra despachos, con datos inventados.

In [ ]:
# Solo en Colab o Jupyter: baja los CSV si no están en la carpeta.
import os, urllib.request

BASE = 'https://raw.githubusercontent.com/fjaltimiras/datacademy-labs/main/claude-code-workshop/05-laboratorio-1/'
for archivo in ['pedidos.csv', 'despachos.csv']:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(BASE + archivo, archivo)

## 1. Cruzar los archivos

In [ ]:
import pandas as pd

pedidos = pd.read_csv('pedidos.csv')
despachos = pd.read_csv('despachos.csv')
cruce = pedidos.merge(despachos, on='pedido', how='outer', suffixes=('_pedidas', '_despachadas'), indicator=True)
print(len(pedidos), 'pedidos ·', len(despachos), 'despachos')
cruce.head()

## 2. Clasificar

In [ ]:
def tipo(f):
    if f['_merge'] == 'left_only': return 'sin_despacho'
    if f['_merge'] == 'right_only': return 'sin_pedido'
    return 'igual' if f['unidades_pedidas'] == f['unidades_despachadas'] else 'cantidad_distinta'

cruce['tipo'] = cruce.apply(tipo, axis=1)
diferencias = cruce[cruce['tipo'] != 'igual']
diferencias[['pedido', 'unidades_pedidas', 'unidades_despachadas', 'tipo']]

## 3. Tu comprobación

Cuenta cada tipo en la tabla y reemplaza los ceros.

In [ ]:
mis_conteos = {'sin_despacho': 0, 'cantidad_distinta': 0, 'sin_pedido': 0}

if mis_conteos == diferencias['tipo'].value_counts().to_dict():
    print('Correcto: los tres conteos cuadran.')
else:
    print('Todavía no coincide. Cuenta otra vez por tipo.')

## 4. Con Claude Code (opcional)

Carga tu `diferencias.csv`, con columnas `pedido` y `tipo`, y ejecuta.

In [ ]:
import os

if os.path.exists('diferencias.csv'):
    mio = pd.read_csv('diferencias.csv')
    faltan_columnas = {'pedido', 'tipo'} - set(mio.columns)
    if faltan_columnas:
        print('Faltan columnas:', faltan_columnas)
    else:
        esperado = set(zip(diferencias['pedido'], diferencias['tipo']))
        obtenido = set(zip(mio['pedido'], mio['tipo']))
        print('Filas repetidas:', int(mio.duplicated(['pedido', 'tipo']).sum()))
        print('Te faltan:', len(esperado - obtenido), '· te sobran:', len(obtenido - esperado))
else:
    print('Aún no cargas diferencias.csv. Este paso es opcional.')